In [3]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os
import time 
import json
import pprint
import importlib

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import signal

import random
import numpy as np
np.set_printoptions(precision=4)
 
from utils import load_data
# from utils import parser, grader2
from math_verify import parse, verify

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, TimeoutError, as_completed
from tqdm import tqdm

from sal.config import Config

In [18]:
# class TimeoutException(Exception):
#     pass

# def timeout_handler(signum, frame):
#     raise TimeoutException()

# def run_with_timeout(completion, gt_answer, timeout=2):
#     # Set the signal handler for SIGALRM
#     signal.signal(signal.SIGALRM, timeout_handler)
#     signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
#     try:
#         c_answer = parse(completion) 
#         # print(c_answer)
#         result = verify(gold=gt_answer, target=c_answer)
#     except TimeoutException:
#         print(f"Timeout: {completion}")
#         c_answer = None
#         result = None
#     finally:
#         signal.alarm(0)  # Cancel alarm if function returns early
#     return c_answer, result

# def compute_samples2correct_onetrial(dataset_orig, result_dir, config_name, trial_idx, max_workers=16, timeout=2):

#     # dataset = load_dataset(dataset_name, name=config_name, split=dataset_split, cache_dir=data_dir)
#     # dataset = load_dataset("json", data_files = data_dir, split='train')
#     # dataset_by_level = dataset.filter(lambda example: example['level'] == level)

#     with open(f"{result_dir}/generate_{config_name}--trial-{trial_idx:03d}.jsonl", 'r', encoding='utf-8') as fin:
#         gen_results = json.load(fin)

#     num_questions = len(dataset_orig)
#     # num_questions = 2

#     samples2correct = np.zeros(num_questions)
#     # for q_idx, data in enumerate(dataset_orig):
#     for q_idx in range(num_questions):
        
#         gt_answer = parse(dataset_orig[q_idx]['solution'], parsing_timeout=None)
#         # gt_answer = dataset_orig[q_idx]['answer']
#         # print(q_idx, gt_answer)

#         q_completions = gen_results["completions"][q_idx]
#         q_cnt_corrects = 0
#         for cidx, completion in enumerate(q_completions):
#             # c_answer, is_correct = run_with_timeout(completion, gt_answer)
#             ## using math-verify functions parse and verify, if error returns to run_with_timeout function
#             c_answer = parse(completion)
#             # print(c_answer)
#             is_correct = verify(gold=gt_answer, target=c_answer) 
#             if is_correct is True: 
#                 # print(c_answer, is_correct)
#                 q_cnt_corrects += 1

#         # print(q_cnt_corrects)
#         samples2correct[q_idx] = (len(q_completions)+1)/(q_cnt_corrects+1)
    
#     print(samples2correct)
#     return samples2correct

def evaluate_completion(completion, gt_answer):
    """
    Worker function to process a single completion.
    We removed the signal logic here because ThreadPoolExecutor handles timeouts natively.
    """
    c_answer = parse(completion) 
    is_correct = verify(gold=gt_answer, target=c_answer)
    return is_correct

def compute_samples2correct_onetrial(dataset_orig, result_dir, config_name, trial_idx, max_workers=16, timeout=2):

    # Load generation results once
    with open(f"{result_dir}/generate_{config_name}--trial-{trial_idx:03d}.jsonl", 'r', encoding='utf-8') as fin:
        gen_results = json.load(fin)
    
    num_questions = len(dataset_orig)
    # num_questions = 2
    
    samples2correct = np.zeros(num_questions)
    
    # Arrays to track totals since tasks will complete out of order
    completions_len = np.zeros(num_questions)
    cnt_corrects = np.zeros(num_questions)

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Dictionary to map running threads (futures) back to their specific question
        future_to_meta = {}

        # 1. Submit all tasks to the pool immediately
        for q_idx in range(num_questions):
            gt_answer = parse(dataset_orig[q_idx]['solution'], parsing_timeout=None)
            # gt_answer = dataset_orig[q_idx]['answer']
            # print(q_idx, gt_answer)
            
            q_completions = gen_results["completions"][q_idx]

            completions_len[q_idx] = len(q_completions)
            for c_idx, completion in enumerate(q_completions):
                future = executor.submit(evaluate_completion, completion, gt_answer)
                # Store the q_idx, c_idx and completion string so we know what this future represents
                future_to_meta[future] = (q_idx, c_idx, completion)

        # 2. Process tasks with a progress bar
        # as_completed() yields futures exactly as they finish
        # tqdm() wraps the iterator to show a live progress bar
        for future in tqdm(as_completed(future_to_meta), total=len(future_to_meta), desc="Grading", disable=True):
            q_idx, c_idx, completion = future_to_meta[future]
            try:
                # wait up to `timeout` seconds for this specific evaluation
                is_correct = future.result(timeout=timeout)
                if is_correct:
                    cnt_corrects[q_idx] += 1
            except TimeoutError:
                tqdm.write(f"Timeout: {q_idx, c_idx}")
            except Exception as e:
                tqdm.write(f"Error grading completion: {e}")
    
    # 3. Compute the final metric using your original formula
    for q_idx in range(num_questions):
        q_len = completions_len[q_idx]
        q_corrects = cnt_corrects[q_idx]
        samples2correct[q_idx] = (q_len + 1) / (q_corrects + 1)

    print(samples2correct)
    return samples2correct

    
max_workers = min(16, (os.cpu_count() or 1) * 2)


# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'
# dataset path
ds_name = "prm800k"
ds_split = "test"
ds_dir = base_dir + "/prm800k/math_splits"

# general params
config = Config()
# pprint.pprint(config)
config.agg_strategy = 'last'
config.temperature = 0.5 
config.max_tokens = 2048
config.bs = 256
config.version = "v01_0_0"

level = 4
num_trials = 2

dataset_orig = load_data.load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = len(dataset_orig)

config_name = f"bon--level-{level}--{config.version}--bs-{config.bs}--temp-{config.temperature}"
result_dir = f"results/{ds_name}/bon--level-{level}/{config_name}"
print(f"config_name = {config_name}")

all_samples2correct = []

for trial_idx in range(num_trials):
    samples2correct = compute_samples2correct_onetrial(dataset_orig, result_dir, config_name, trial_idx, max_workers=max_workers)
    all_samples2correct.append(samples2correct)

all_samples2correct = np.concatenate(all_samples2correct)

samples2correct_mean = np.mean(all_samples2correct)
samples2correct_std = np.std(all_samples2correct, ddof=1)/np.sqrt(num_trials*num_questions) # 128 is number of prompts for level 4 

print(
    f"{samples2correct_mean:0.4f} (\u00B1{samples2correct_std:0.4f}), "
)

config_name = bon--level-4--v01_0_0--bs-256--temp-0.5
[ 51.4     12.85     1.2476   3.5205  28.5556   8.0312  85.6667   1.0198
  13.5263 257.       1.8759  36.7143   8.8621 257.       1.218    7.1389
 128.5     23.3636   2.0078 128.5      2.2743   1.2415  85.6667  25.7
   5.4681  51.4      1.9323   2.2155  16.0625  16.0625   2.4476  21.4167
   6.119   64.25    21.4167   2.8876   2.6224   4.7593  18.3571   3.3816
   6.9459  12.85    10.28     5.14     7.1389   7.5588  36.7143   9.1786
   9.8846  12.2381 257.      17.1333 257.     257.      25.7      1.1735
 257.       7.1389   1.0078  64.25   257.     257.      51.4     36.7143
   2.5196   1.0844   4.8491  32.125  257.       2.1966   1.7972  10.28
   7.3429  32.125  257.      28.5556 257.     257.      36.7143  85.6667
  19.7692   5.2449 128.5      1.2723   7.3429   8.2903  36.7143   1.1322
   6.9459  19.7692   1.3247   5.9767   2.7634   2.6771 257.       2.124
  17.1333  28.5556 257.     257.       3.2125   1.0321   6.5897   4.431
  42